In [21]:
# ============================================================
# NOTEBOOK 9 — ROBUST VALIDATION & LEAKAGE DIAGNOSTIC
# CELL 1 — IMPORT LIBRARIES
# ============================================================
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import joblib
import warnings
from pathlib import Path
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import (
    roc_auc_score,
    accuracy_score,
    precision_score,
    recall_score,
    f1_score
)
from imblearn.over_sampling import SMOTENC
from xgboost import XGBClassifier
warnings.filterwarnings("ignore")
print("Libraries imported successfully.")

# ============================================================
# CELL 2 — DEFINE DIRECTORIES
# ============================================================
BASE_DIR = Path.cwd()
PROCESSED_DIR = BASE_DIR / "processed_data"
MODELS_DIR = BASE_DIR / "models"
RESULTS_DIR = BASE_DIR / "results"
PROCESSED_DIR = Path("../data/processed")

MODELS_DIR.mkdir(exist_ok=True)
RESULTS_DIR.mkdir(exist_ok=True)

print("Project directory:", BASE_DIR)
print("Processed data:", PROCESSED_DIR)
print("Models:", MODELS_DIR)
print("Results:", RESULTS_DIR)

Libraries imported successfully.
Project directory: c:\Users\HP\Documents\SP-XGBOOST\SP-Notebooks
Processed data: ..\data\processed
Models: c:\Users\HP\Documents\SP-XGBOOST\SP-Notebooks\models
Results: c:\Users\HP\Documents\SP-XGBOOST\SP-Notebooks\results


In [22]:
# ============================================================
# CELL 3 — LOAD ORIGINAL TRAINING DATA
# ============================================================

X_train_clean = pd.read_csv(
    PROCESSED_DIR / "X_train_clean.csv"
)

y_train_clean = pd.read_csv(
    PROCESSED_DIR / "y_train_clean.csv"
).squeeze()

print("=" * 70)
print("ORIGINAL TRAINING DATA")
print("=" * 70)

print("X_train_clean shape:",
    X_train_clean.shape)

print("y_train_clean shape:",
    y_train_clean.shape)

print("\nTarget distribution:")
print(
    y_train_clean
    .value_counts()
    .sort_index()
)

ORIGINAL TRAINING DATA
X_train_clean shape: (455, 58)
y_train_clean shape: (455,)

Target distribution:
RISK_LABEL
0    329
1    126
Name: count, dtype: int64


In [23]:
# ============================================================
# CELL 4 — LOAD SHAP-SELECTED FEATURES
# ============================================================

selected_features_df = pd.read_csv(
    RESULTS_DIR / "shap_selected_features.csv"
)

selected_features = (
    selected_features_df["Feature"]
    .tolist()
)

print("Number of SHAP-selected features:",
    len(selected_features))

print("\nSelected features:")
for i, feature in enumerate(selected_features, 1):
    print(f"{i:02d}. {feature}")

Number of SHAP-selected features: 47

Selected features:
01. GPA_S1
02. Hopeless/unmotivated
03. Sleep_hrs
04. LAB_AVG
05. Programme prepares for career
06. Concentration in self-study
07. Adequate supervision
08. CA_AVG
09. Financial_diff
10. Employment_hrs
11. Can improve performance
12. Self-motivates
13. Sense of belonging
14. Manages time effectively
15. Early intervention provided
16. Persists when difficult
17. Peer study groups effective
18. Anxious about assessments
19. Takes rest breaks
20. Seeks help when stuck
21. Regular exercise
22. Participates in class
23. Considered break
24. Age_group
25. Understands content pre-exam
26. CLIN_AVG
27. Takes organised notes
28. Physical health affected
29. ATT_RATE
30. Lecturers approachable
31. Sets academic goals
32. Study_hrs_day
33. Completes readings
34. EXAM_AVG
35. Confident in clinical skills
36. Prepared for clinical assess
37. Rotations affect performance
38. Self_risk_percep
39. Schedule conflicts
40. Gender
41. Sleep difficu

In [24]:
# ============================================================
# CELL 5 — APPLY SHAP FEATURE SET TO ORIGINAL TRAINING DATA
# ============================================================

missing_features = [
    feature
    for feature in selected_features
    if feature not in X_train_clean.columns
]

if missing_features:
    print("Missing features:")
    print(missing_features)

    raise ValueError(
        "Some SHAP-selected features are missing."
    )

X_train_selected_original = (
    X_train_clean[selected_features]
    .copy()
)

print("=" * 70)
print("SHAP-SELECTED ORIGINAL TRAINING DATA")
print("=" * 70)

print(
    "X_train_selected_original shape:",
    X_train_selected_original.shape
)

print(
    "y_train_clean shape:",
    y_train_clean.shape
)

SHAP-SELECTED ORIGINAL TRAINING DATA
X_train_selected_original shape: (455, 47)
y_train_clean shape: (455,)


In [25]:
# ============================================================
# CELL 6 — DATA QUALITY CHECK
# ============================================================

print("=" * 70)
print("DATA QUALITY CHECK")
print("=" * 70)

print(
    "Missing values:",
    X_train_selected_original.isnull().sum().sum()
)

print(
    "Duplicate columns:",
    X_train_selected_original.columns.duplicated().sum()
)

print("\nData types:")
print(
    X_train_selected_original.dtypes
    .value_counts()
)

DATA QUALITY CHECK
Missing values: 0
Duplicate columns: 0

Data types:
int64      41
float64     6
Name: count, dtype: int64


In [26]:
# ============================================================
# CELL 7
# IDENTIFY ACTUAL CATEGORICAL VARIABLES
# ============================================================

# These categorical variables are confirmed from your dataset.
# Programme is deliberately NOT included because it does not
# exist in X_train_original.

categorical_columns = [
    "ASSIGN_LATE",
    "Age_group",
    "Gender",
    "Year_study",
    "SES",
    "Financial_diff",
    "Self_risk_percep"
]

# ------------------------------------------------------------
# VERIFY THAT ALL SPECIFIED CATEGORICAL COLUMNS EXIST
# ------------------------------------------------------------

missing_categorical = [
    col
    for col in categorical_columns
    if col not in X_train_clean.columns
]

if missing_categorical:
    raise KeyError(
        f"These categorical columns are missing: "
        f"{missing_categorical}"
    )

# ------------------------------------------------------------
# IDENTIFY NUMERICAL VARIABLES
# ------------------------------------------------------------

numerical_columns = [
    col
    for col in X_train_clean.columns
    if col not in categorical_columns
]

# ------------------------------------------------------------
# DISPLAY RESULTS
# ------------------------------------------------------------

print("=" * 80)
print("NOTEBOOK 9 — FEATURE TYPE IDENTIFICATION")
print("=" * 80)

print(
    "Total predictors:",
    len(X_train_clean.columns)
)

print(
    "Categorical predictors:",
    len(categorical_columns)
)

print(
    "Numerical predictors:",
    len(numerical_columns)
)

print("\nCategorical predictors:")

for i, col in enumerate(
    categorical_columns,
    start=1
):
    print(f"{i}. {col}")

print("\nNumerical predictors:")

for i, col in enumerate(
    numerical_columns,
    start=1
):
    print(f"{i}. {col}")

print("\n✓ All categorical columns verified.")

NOTEBOOK 9 — FEATURE TYPE IDENTIFICATION
Total predictors: 58
Categorical predictors: 7
Numerical predictors: 51

Categorical predictors:
1. ASSIGN_LATE
2. Age_group
3. Gender
4. Year_study
5. SES
6. Financial_diff
7. Self_risk_percep

Numerical predictors:
1. GPA_S1
2. CA_AVG
3. EXAM_AVG
4. CLIN_AVG
5. LAB_AVG
6. ATT_RATE
7. Employment_hrs
8. Study_hrs_day
9. Sleep_hrs
10. Reviews notes within 24h
11. Understands content pre-exam
12. Seeks help when stuck
13. Uses library regularly
14. Completes readings
15. Takes organised notes
16. Concentration in self-study
17. Participates in class
18. Clinical takes study time
19. Prepared for clinical assess
20. Rotations affect performance
21. Adequate supervision
22. Schedule conflicts
23. Confident in clinical skills
24. Anxious about assessments
25. Sleep difficulty
26. Burnt out
27. Hopeless/unmotivated
28. Physical health affected
29. Considered break
30. Emotionally supported
31. Lecturers approachable
32. Sleep affects concentration
33.

In [27]:
# ============================================================
# CELL 8
# PREPARE DATA FOR SMOTENC
# ============================================================

X_cv = X_train_clean.copy()

# ------------------------------------------------------------
# Ensure categorical variables are integer encoded
# ------------------------------------------------------------

for col in categorical_columns:

    X_cv[col] = pd.to_numeric(
        X_cv[col],
        errors="raise"
    ).astype(int)

# ------------------------------------------------------------
# Ensure numerical variables are numeric
# ------------------------------------------------------------

for col in numerical_columns:

    X_cv[col] = pd.to_numeric(
        X_cv[col],
        errors="raise"
    )

# ------------------------------------------------------------
# Check missing values
# ------------------------------------------------------------

missing_total = X_cv.isnull().sum().sum()

print("=" * 80)
print("NOTEBOOK 9 — SMOTENC DATA PREPARATION")
print("=" * 80)

print("Shape:", X_cv.shape)

print(
    "Missing values:",
    missing_total
)

print(
    "Categorical variables:",
    len(categorical_columns)
)

print(
    "Numerical variables:",
    len(numerical_columns)
)

if missing_total == 0:
    print("\n✓ No missing values detected.")
else:
    print("\n⚠ Missing values detected.")

NOTEBOOK 9 — SMOTENC DATA PREPARATION
Shape: (455, 58)
Missing values: 0
Categorical variables: 7
Numerical variables: 51

✓ No missing values detected.


In [28]:
# ============================================================
# NOTEBOOK 9 — CELL 9
# VERIFY SMOTENC CATEGORICAL INDICES
# ============================================================

categorical_indices = [
    X_cv.columns.get_loc(col)
    for col in categorical_columns
]

print("=" * 80)
print("SMOTENC CATEGORICAL INDICES")
print("=" * 80)

print(
    "Number of categorical columns:",
    len(categorical_columns)
)

print(
    "Categorical indices:",
    categorical_indices
)

print("\nColumn → Index:")

for col in categorical_columns:

    index = X_cv.columns.get_loc(col)

    print(
        f"{col:25s} → {index}"
    )

print("\n✓ SMOTENC categorical indices created successfully.")

SMOTENC CATEGORICAL INDICES
Number of categorical columns: 7
Categorical indices: [6, 7, 8, 9, 10, 11, 15]

Column → Index:
ASSIGN_LATE               → 6
Age_group                 → 7
Gender                    → 8
Year_study                → 9
SES                       → 10
Financial_diff            → 11
Self_risk_percep          → 15

✓ SMOTENC categorical indices created successfully.


In [29]:
# ============================================================
# CELL 10 — DEFINE 5-FOLD STRATIFIED CV
# ============================================================

skf = StratifiedKFold(
    n_splits=5,
    shuffle=True,
    random_state=42
)

print("5-fold Stratified Cross-Validation created.")
print("Number of folds:", skf.get_n_splits())

5-fold Stratified Cross-Validation created.
Number of folds: 5


In [30]:
# ============================================================
# CELL 11 — LOAD BEST OPTUNA PARAMETERS
# ============================================================

best_params_df = pd.read_csv(
    RESULTS_DIR / "SP_XGBoost_best_parameters.csv"
)

best_params = best_params_df.iloc[0].to_dict()

# Remove validation score from model parameters
best_params.pop(
    "best_validation_ROC_AUC",
    None
)

# Convert numerical parameters where necessary
for key in [
    "n_estimators",
    "max_depth",
    "min_child_weight"
]:
    if key in best_params:
        best_params[key] = int(best_params[key])

print("=" * 70)
print("BEST OPTUNA PARAMETERS")
print("=" * 70)

for key, value in best_params.items():
    print(f"{key}: {value}")

BEST OPTUNA PARAMETERS
n_estimators: 450
max_depth: 8
learning_rate: 0.0314210110072907
min_child_weight: 1
subsample: 0.7835513171100621
colsample_bytree: 0.8432983574665387
gamma: 0.0018324957139797
reg_alpha: 0.0901843928610209
reg_lambda: 7.110933538908288e-08


In [31]:
# ============================================================
# CELL 12 — PREPARE CV RESULTS
# ============================================================
cv_results = []

print("CV results storage initialized.")

CV results storage initialized.


In [32]:
# ============================================================
# CELL 13 --ROBUST FOLD-WISE SMOTENC + SP-XGBOOST
# ============================================================

# ------------------------------------------------------------
# IMPORTANT:
# Recreate categorical indices here.
# Do NOT rely on an older categorical_indices variable.
# ------------------------------------------------------------

categorical_columns = [
    "ASSIGN_LATE",
    "Age_group",
    "Gender",
    "Year_study",
    "SES",
    "Financial_diff",
    "Self_risk_percep"
]

# Verify columns exist
missing_columns = [
    col
    for col in categorical_columns
    if col not in X_cv.columns
]

if missing_columns:
    raise KeyError(
        f"Missing categorical columns: {missing_columns}"
    )

# Recreate categorical indices
smotenc_indices = [
    X_cv.columns.get_loc(col)
    for col in categorical_columns
]

print("=" * 80)
print("NOTEBOOK 9 — SMOTENC CONFIGURATION")
print("=" * 80)

print("Categorical columns:")
for col in categorical_columns:
    print(" -", col)

print("\nCategorical indices:")
print(smotenc_indices)

print(
    "\nNumber of categorical features:",
    len(smotenc_indices)
)

# Safety check
if len(smotenc_indices) == 0:
    raise ValueError(
        "SMOTENC categorical indices are empty. "
        "Stop before resampling."
    )

# ------------------------------------------------------------
# Cross-validation
# ------------------------------------------------------------

cv_results = []

for fold, (train_idx, valid_idx) in enumerate(
    skf.split(X_cv, y_train_clean),
    start=1
):

    print("\n" + "=" * 80)
    print(f"FOLD {fold}")
    print("=" * 80)

    # --------------------------------------------------------
    # 1. Split original training data
    # --------------------------------------------------------

    X_fold_train = X_cv.iloc[
        train_idx
    ].copy()

    X_fold_valid = X_cv.iloc[
        valid_idx
    ].copy()

    y_fold_train = y_train_clean.iloc[
        train_idx
    ].copy()

    y_fold_valid = y_train_clean.iloc[
        valid_idx
    ].copy()

    print(
        "Training before SMOTENC:",
        X_fold_train.shape
    )

    print(
        "Validation:",
        X_fold_valid.shape
    )

    print("\nClass distribution BEFORE SMOTENC:")
    print(
        y_fold_train.value_counts().sort_index()
    )

    # --------------------------------------------------------
    # 2. Create SMOTENC INSIDE each fold
    # --------------------------------------------------------

    smote_nc = SMOTENC(
        categorical_features=smotenc_indices,
        random_state=42,
        k_neighbors=5
    )

    # Safety check before fitting
    print(
        "\nSMOTENC categorical_features:",
        smote_nc.categorical_features
    )

    # --------------------------------------------------------
    # 3. Apply SMOTENC ONLY to fold training data
    # --------------------------------------------------------

    X_fold_resampled, y_fold_resampled = (
        smote_nc.fit_resample(
            X_fold_train,
            y_fold_train
        )
    )

    print(
        "\nTraining AFTER SMOTENC:",
        X_fold_resampled.shape
    )

    print(
        "Class distribution AFTER SMOTENC:"
    )

    print(
        pd.Series(
            y_fold_resampled
        ).value_counts().sort_index()
    )

    # --------------------------------------------------------
    # 4. Convert to DataFrame
    # --------------------------------------------------------

    X_fold_resampled = pd.DataFrame(
        X_fold_resampled,
        columns=X_cv.columns
    )

    X_fold_valid = pd.DataFrame(
        X_fold_valid,
        columns=X_cv.columns
    )

    # --------------------------------------------------------
    # 5. Apply SHAP-selected 47 features
    # --------------------------------------------------------

    X_fold_resampled_shap = (
        X_fold_resampled[
            selected_features
        ].copy()
    )

    X_fold_valid_shap = (
        X_fold_valid[
            selected_features
        ].copy()
    )

    print(
        "\nSHAP-selected training:",
        X_fold_resampled_shap.shape
    )

    print(
        "SHAP-selected validation:",
        X_fold_valid_shap.shape
    )

    # --------------------------------------------------------
    # 6. Train SP-XGBoost
    # --------------------------------------------------------

    model = XGBClassifier(
        **best_params,
        objective="binary:logistic",
        eval_metric="logloss",
        random_state=42,
        n_jobs=-1
    )

    model.fit(
        X_fold_resampled_shap,
        y_fold_resampled
    )

    # --------------------------------------------------------
    # 7. Predict on untouched validation fold
    # --------------------------------------------------------

    fold_probability = model.predict_proba(
        X_fold_valid_shap
    )[:, 1]

    fold_prediction = model.predict(
        X_fold_valid_shap
    )

    # --------------------------------------------------------
    # 8. Calculate metrics
    # --------------------------------------------------------

    fold_auc = roc_auc_score(
        y_fold_valid,
        fold_probability
    )

    fold_accuracy = accuracy_score(
        y_fold_valid,
        fold_prediction
    )

    fold_precision = precision_score(
        y_fold_valid,
        fold_prediction,
        zero_division=0
    )

    fold_recall = recall_score(
        y_fold_valid,
        fold_prediction,
        zero_division=0
    )

    fold_f1 = f1_score(
        y_fold_valid,
        fold_prediction,
        zero_division=0
    )

    cv_results.append({
        "Fold": fold,
        "ROC_AUC": fold_auc,
        "Accuracy": fold_accuracy,
        "Precision": fold_precision,
        "Recall": fold_recall,
        "F1": fold_f1
    })

    print("\nFold results:")
    print(f"ROC-AUC  : {fold_auc:.4f}")
    print(f"Accuracy : {fold_accuracy:.4f}")
    print(f"Precision: {fold_precision:.4f}")
    print(f"Recall   : {fold_recall:.4f}")
    print(f"F1       : {fold_f1:.4f}")


# ------------------------------------------------------------
# FINAL CV SUMMARY
# ------------------------------------------------------------

cv_results_df = pd.DataFrame(cv_results)

print("\n" + "=" * 80)
print("NOTEBOOK 9 — CROSS-VALIDATION SUMMARY")
print("=" * 80)

print(cv_results_df)

print("\nMean ROC-AUC:")
print(
    f"{cv_results_df['ROC_AUC'].mean():.4f}"
)

print("\nMean Accuracy:")
print(
    f"{cv_results_df['Accuracy'].mean():.4f}"
)

print("\nMean Precision:")
print(
    f"{cv_results_df['Precision'].mean():.4f}"
)

print("\nMean Recall:")
print(
    f"{cv_results_df['Recall'].mean():.4f}"
)

print("\nMean F1:")
print(
    f"{cv_results_df['F1'].mean():.4f}"
)

print("\n✓ NOTEBOOK 9 CELL 13 COMPLETED")

NOTEBOOK 9 — SMOTENC CONFIGURATION
Categorical columns:
 - ASSIGN_LATE
 - Age_group
 - Gender
 - Year_study
 - SES
 - Financial_diff
 - Self_risk_percep

Categorical indices:
[6, 7, 8, 9, 10, 11, 15]

Number of categorical features: 7

FOLD 1
Training before SMOTENC: (364, 58)
Validation: (91, 58)

Class distribution BEFORE SMOTENC:
RISK_LABEL
0    264
1    100
Name: count, dtype: int64

SMOTENC categorical_features: [6, 7, 8, 9, 10, 11, 15]

Training AFTER SMOTENC: (528, 58)
Class distribution AFTER SMOTENC:
RISK_LABEL
0    264
1    264
Name: count, dtype: int64

SHAP-selected training: (528, 47)
SHAP-selected validation: (91, 47)

Fold results:
ROC-AUC  : 0.6391
Accuracy : 0.7143
Precision: 0.5000
Recall   : 0.4231
F1       : 0.4583

FOLD 2
Training before SMOTENC: (364, 58)
Validation: (91, 58)

Class distribution BEFORE SMOTENC:
RISK_LABEL
0    263
1    101
Name: count, dtype: int64

SMOTENC categorical_features: [6, 7, 8, 9, 10, 11, 15]

Training AFTER SMOTENC: (526, 58)
Class dis

In [33]:
# ============================================================
# CELL 14 -CROSS-VALIDATION PERFORMANCE SUMMARY
# ============================================================

print("=" * 80)
print("NOTEBOOK 9 — SP-XGBOOST CROSS-VALIDATION SUMMARY")
print("=" * 80)

# ------------------------------------------------------------
# Display fold-by-fold results
# ------------------------------------------------------------

print("\nFold-by-fold performance:")
display(
    cv_results_df.round(4)
)

# ------------------------------------------------------------
# Calculate mean and standard deviation
# ------------------------------------------------------------

metrics = [
    "ROC_AUC",
    "Accuracy",
    "Precision",
    "Recall",
    "F1"
]

cv_summary = pd.DataFrame({
    "Metric": metrics,
    "Mean": [
        cv_results_df[m].mean()
        for m in metrics
    ],
    "Std": [
        cv_results_df[m].std()
        for m in metrics
    ]
})

print("\n" + "=" * 80)
print("MEAN ± STANDARD DEVIATION")
print("=" * 80)

display(
    cv_summary.round(4)
)

# ------------------------------------------------------------
# Save results
# ------------------------------------------------------------

cv_results_df.to_csv(
    PROCESSED_DIR / "SP_XGBoost_CV_fold_results.csv",
    index=False
)

cv_summary.to_csv(
    PROCESSED_DIR / "SP_XGBoost_CV_summary.csv",
    index=False
)

print(
    "\n✓ CV results saved successfully."
)

NOTEBOOK 9 — SP-XGBOOST CROSS-VALIDATION SUMMARY

Fold-by-fold performance:


,Fold,ROC_AUC,Accuracy,Precision,Recall,F1
0,1,0.6391,0.7143,0.5000,0.4231,0.4583
1,2,0.6582,0.7363,0.5263,0.4000,0.4545
2,3,0.6024,0.6703,0.3810,0.3200,0.3478
3,4,0.7424,0.7473,0.6000,0.2400,0.3429
4,5,0.5582,0.6264,0.3200,0.3200,0.3200



MEAN ± STANDARD DEVIATION


,Metric,Mean,Std
0,ROC_AUC,0.6401,0.0688
1,Accuracy,0.6989,0.0501
2,Precision,0.4655,0.1132
3,Recall,0.3406,0.0730
4,F1,0.3847,0.0663



✓ CV results saved successfully.


In [34]:
# ============================================================
# NOTEBOOK 9 — CELL 15
# MODEL PERFORMANCE COMPARISON
# ============================================================

print("=" * 80)
print("MODEL PERFORMANCE COMPARISON")
print("=" * 80)

# ------------------------------------------------------------
# Existing model results
# ------------------------------------------------------------

model_comparison = pd.DataFrame({

    "Model": [
        "Baseline XGBoost",
        "S-XGBoost",
        "SP-XGBoost (Previous Test)",
        "SP-XGBoost (5-Fold CV)"
    ],

    "Accuracy": [
        0.7105,
        0.7105,
        0.7018,
        cv_results_df["Accuracy"].mean()
    ],

    "Precision": [
        0.4583,
        0.4615,
        0.4400,
        cv_results_df["Precision"].mean()
    ],

    "Recall": [
        0.3548,
        0.3871,
        0.3548,
        cv_results_df["Recall"].mean()
    ],

    "F1": [
        0.4000,
        0.4211,
        0.3929,
        cv_results_df["F1"].mean()
    ],

    "ROC_AUC": [
        0.6502,
        0.6421,
        0.6319,
        cv_results_df["ROC_AUC"].mean()
    ]
})

# ------------------------------------------------------------
# Display comparison
# ------------------------------------------------------------

display(
    model_comparison.round(4)
)

# ------------------------------------------------------------
# Determine best model by ROC-AUC
# ------------------------------------------------------------

best_auc_idx = model_comparison[
    "ROC_AUC"
].idxmax()

best_auc_model = model_comparison.loc[
    best_auc_idx,
    "Model"
]

best_auc = model_comparison.loc[
    best_auc_idx,
    "ROC_AUC"
]

# ------------------------------------------------------------
# Determine best model by F1
# ------------------------------------------------------------

best_f1_idx = model_comparison[
    "F1"
].idxmax()

best_f1_model = model_comparison.loc[
    best_f1_idx,
    "Model"
]

best_f1 = model_comparison.loc[
    best_f1_idx,
    "F1"
]

print("\n" + "=" * 80)
print("BEST PERFORMANCE")
print("=" * 80)

print(
    f"Best ROC-AUC: {best_auc_model} "
    f"({best_auc:.4f})"
)

print(
    f"Best F1: {best_f1_model} "
    f"({best_f1:.4f})"
)

# ------------------------------------------------------------
# Save comparison
# ------------------------------------------------------------

model_comparison.to_csv(
    PROCESSED_DIR / "model_performance_comparison.csv",
    index=False
)

print(
    "\n✓ Model comparison saved successfully."
)

MODEL PERFORMANCE COMPARISON


,Model,Accuracy,Precision,Recall,F1,ROC_AUC
0,Baseline XGBoost,0.7105,0.4583,0.3548,0.4000,0.6502
1,S-XGBoost,0.7105,0.4615,0.3871,0.4211,0.6421
2,SP-XGBoost (Previous Test),0.7018,0.4400,0.3548,0.3929,0.6319
3,SP-XGBoost (5-Fold CV),0.6989,0.4655,0.3406,0.3847,0.6401



BEST PERFORMANCE
Best ROC-AUC: Baseline XGBoost (0.6502)
Best F1: S-XGBoost (0.4211)

✓ Model comparison saved successfully.


In [35]:
# ==========================================================
# CELL 16 95% CONFIDENCE INTERVALS FOR SP-XGBOOST CV
# ============================================================

print("=" * 80)
print("NOTEBOOK 9 — 95% CONFIDENCE INTERVALS")
print("=" * 80)

import numpy as np
from scipy import stats

confidence_level = 0.95
alpha = 1 - confidence_level

metrics = [
    "ROC_AUC",
    "Accuracy",
    "Precision",
    "Recall",
    "F1"
]

ci_results = []

n_folds = len(cv_results_df)

for metric in metrics:

    values = cv_results_df[metric].values

    mean_value = np.mean(values)
    std_value = np.std(values, ddof=1)

    standard_error = (
        std_value / np.sqrt(n_folds)
    )

    t_critical = stats.t.ppf(
        1 - alpha / 2,
        df=n_folds - 1
    )

    margin_error = (
        t_critical * standard_error
    )

    lower_ci = mean_value - margin_error
    upper_ci = mean_value + margin_error

    ci_results.append({
        "Metric": metric,
        "Mean": mean_value,
        "Std": std_value,
        "95% CI Lower": lower_ci,
        "95% CI Upper": upper_ci
    })

ci_df = pd.DataFrame(ci_results)

# ------------------------------------------------------------
# Display
# ------------------------------------------------------------

display(
    ci_df.round(4)
)

# ------------------------------------------------------------
# Save
# ------------------------------------------------------------

ci_df.to_csv(
    PROCESSED_DIR / "SP_XGBoost_CV_95CI.csv",
    index=False
)

print("\n" + "=" * 80)
print("KEY SP-XGBOOST RESULT")
print("=" * 80)

auc_row = ci_df[
    ci_df["Metric"] == "ROC_AUC"
].iloc[0]

print(
    f"Mean ROC-AUC: "
    f"{auc_row['Mean']:.4f}"
)

print(
    f"95% CI: "
    f"{auc_row['95% CI Lower']:.4f} "
    f"to "
    f"{auc_row['95% CI Upper']:.4f}"
)

print("\n✓ 95% confidence intervals calculated.")

NOTEBOOK 9 — 95% CONFIDENCE INTERVALS


,Metric,Mean,Std,95% CI Lower,95% CI Upper
0,ROC_AUC,0.6401,0.0688,0.5546,0.7255
1,Accuracy,0.6989,0.0501,0.6367,0.7611
2,Precision,0.4655,0.1132,0.3248,0.6061
3,Recall,0.3406,0.0730,0.2500,0.4312
4,F1,0.3847,0.0663,0.3024,0.4671



KEY SP-XGBOOST RESULT
Mean ROC-AUC: 0.6401
95% CI: 0.5546 to 0.7255

✓ 95% confidence intervals calculated.


In [36]:
# ============================================================
# NOTEBOOK 9 — CELL 17
# INVESTIGATE OPTUNA VALIDATION VS CROSS-VALIDATION GAP
# ============================================================

print("=" * 80)
print("NOTEBOOK 9 — VALIDATION GAP DIAGNOSTIC")
print("=" * 80)

print("\nPrevious Optuna internal validation ROC-AUC:")
print("0.9410")

print("\nCurrent 5-fold SP-XGBoost CV ROC-AUC:")
print(
    f"{cv_results_df['ROC_AUC'].mean():.4f}"
)

print("\nDifference:")
print(
    f"{0.9410 - cv_results_df['ROC_AUC'].mean():.4f}"
)

print("\n" + "=" * 80)
print("CV FOLD ROC-AUC VALUES")
print("=" * 80)

for _, row in cv_results_df.iterrows():
    print(
        f"Fold {int(row['Fold'])}: "
        f"{row['ROC_AUC']:.4f}"
    )

print("\n" + "=" * 80)
print("OPTUNA BEST PARAMETERS")
print("=" * 80)

print(best_params)

print("\n✓ Validation gap diagnostic completed.")

NOTEBOOK 9 — VALIDATION GAP DIAGNOSTIC

Previous Optuna internal validation ROC-AUC:
0.9410

Current 5-fold SP-XGBoost CV ROC-AUC:
0.6401

Difference:
0.3009

CV FOLD ROC-AUC VALUES
Fold 1: 0.6391
Fold 2: 0.6582
Fold 3: 0.6024
Fold 4: 0.7424
Fold 5: 0.5582

OPTUNA BEST PARAMETERS
{'n_estimators': 450, 'max_depth': 8, 'learning_rate': 0.0314210110072907, 'min_child_weight': 1, 'subsample': 0.7835513171100621, 'colsample_bytree': 0.8432983574665387, 'gamma': 0.0018324957139797, 'reg_alpha': 0.0901843928610209, 'reg_lambda': 7.110933538908288e-08}

✓ Validation gap diagnostic completed.


In [37]:
# ============================================================
# NOTEBOOK 9 — CELL 18
# VERIFY SP-XGBOOST FEATURE PIPELINE
# ============================================================

print("=" * 80)
print("NOTEBOOK 9 — SP-XGBOOST PIPELINE CONSISTENCY CHECK")
print("=" * 80)

# ------------------------------------------------------------
# 1. Check SHAP-selected features
# ------------------------------------------------------------

print("\nSHAP-selected feature count:")
print(len(selected_features))

print("\nExpected:")
print(47)

if len(selected_features) == 47:
    print("✓ Correct: 47 SHAP-selected features.")
else:
    print(
        f"⚠ WARNING: {len(selected_features)} "
        "features found instead of 47."
    )

# ------------------------------------------------------------
# 2. Check whether all selected features exist
# ------------------------------------------------------------

missing_selected = [
    col
    for col in selected_features
    if col not in X_cv.columns
]

print("\nMissing SHAP-selected features:")
print(missing_selected)

if len(missing_selected) == 0:
    print("✓ All SHAP-selected features exist.")
else:
    print("⚠ Some selected features are missing.")

# ------------------------------------------------------------
# 3. Check feature ordering
# ------------------------------------------------------------

print("\nNumber of columns in X_cv:")
print(X_cv.shape[1])

print("\nNumber of selected features:")
print(len(selected_features))

# ------------------------------------------------------------
# 4. Check data types of selected features
# ------------------------------------------------------------

selected_dtypes = X_cv[
    selected_features
].dtypes

print("\nSHAP-selected feature data types:")
print(selected_dtypes)

# ------------------------------------------------------------
# 5. Check missing values
# ------------------------------------------------------------

selected_missing = (
    X_cv[selected_features]
    .isnull()
    .sum()
    .sum()
)

print("\nMissing values among SHAP-selected features:")
print(selected_missing)

# ------------------------------------------------------------
# 6. Check target
# ------------------------------------------------------------

print("\nTarget variable:")
print("y_train_original")

print("\nTarget shape:")
print(y_train_clean.shape)

print("\nTarget distribution:")
print(
    y_train_clean
    .value_counts()
    .sort_index()
)

# ------------------------------------------------------------
# 7. Verify best parameters
# ------------------------------------------------------------

print("\n" + "=" * 80)
print("OPTUNA PARAMETERS USED IN SP-XGBOOST")
print("=" * 80)

for parameter, value in best_params.items():
    print(
        f"{parameter}: {value}"
    )

print("\n✓ Pipeline consistency check completed.")

NOTEBOOK 9 — SP-XGBOOST PIPELINE CONSISTENCY CHECK

SHAP-selected feature count:
47

Expected:
47
✓ Correct: 47 SHAP-selected features.

Missing SHAP-selected features:
[]
✓ All SHAP-selected features exist.

Number of columns in X_cv:
58

Number of selected features:
47

SHAP-selected feature data types:
GPA_S1                           float64
Hopeless/unmotivated               int64
Sleep_hrs                          int64
LAB_AVG                          float64
Programme prepares for career      int64
Concentration in self-study        int64
Adequate supervision               int64
CA_AVG                           float64
Financial_diff                     int64
Employment_hrs                     int64
Can improve performance            int64
Self-motivates                     int64
Sense of belonging                 int64
Manages time effectively           int64
Early intervention provided        int64
Persists when difficult            int64
Peer study groups effective        in

In [38]:
# ============================================================
# NOTEBOOK 9 — CELL 19
# REPRODUCE OPTUNA VALIDATION PERFORMANCE
# ============================================================

# ============================================================
# NOTEBOOK 9 — CELL 19
# REPRODUCE OPTUNA VALIDATION PERFORMANCE
# ============================================================
# Required imports
import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split
from sklearn.metrics import (
    roc_auc_score,
    accuracy_score,
    precision_score,
    recall_score,
    f1_score
)
from xgboost import XGBClassifier

print("=" * 80)
print("NOTEBOOK 9 — REPRODUCING OPTUNA VALIDATION SCORE")
print("=" * 80)

# ------------------------------------------------------------
# 1. LOAD SMOTENC-BALANCED TRAINING DATA
# ------------------------------------------------------------

X_smotenc_path = (
    PROCESSED_DIR / "X_train_smotenc.csv"
)

y_smotenc_path = (
    PROCESSED_DIR / "y_train_smotenc.csv"
)

print("\nLoading:")
print(X_smotenc_path)
print(y_smotenc_path)

X_balanced = pd.read_csv(
    X_smotenc_path
)

y_balanced = pd.read_csv(
    y_smotenc_path
)

# Convert target DataFrame to Series
if isinstance(y_balanced, pd.DataFrame):
    y_balanced = y_balanced.iloc[:, 0]

print("\nBalanced training shape:")
print(X_balanced.shape)

print("\nBalanced class distribution:")
print(
    y_balanced.value_counts()
    .sort_index()
)

# ------------------------------------------------------------
# 2. VERIFY THAT ALL 47 SHAP FEATURES EXIST
# ------------------------------------------------------------

missing_selected = [
    col
    for col in selected_features
    if col not in X_balanced.columns
]

if missing_selected:
    raise KeyError(
        f"Missing SHAP-selected features: "
        f"{missing_selected}"
    )

print(
    "\n✓ All 47 SHAP-selected features "
    "are present."
)

# ------------------------------------------------------------
# 3. APPLY THE 47 SHAP-SELECTED FEATURES
# ------------------------------------------------------------

X_balanced_shap = X_balanced[
    selected_features
].copy()

print(
    "\nSHAP-selected balanced training shape:"
)

print(
    X_balanced_shap.shape
)

# ------------------------------------------------------------
# 4. CREATE INTERNAL VALIDATION SPLIT
# ------------------------------------------------------------

X_opt_train, X_opt_valid, y_opt_train, y_opt_valid = (
    train_test_split(
        X_balanced_shap,
        y_balanced,
        test_size=0.20,
        stratify=y_balanced,
        random_state=42
    )
)

print("\nOptuna-style training shape:")
print(X_opt_train.shape)

print(
    "Optuna-style validation shape:"
)

print(X_opt_valid.shape)

print("\nTraining class distribution:")
print(
    y_opt_train.value_counts()
    .sort_index()
)

print("\nValidation class distribution:")
print(
    y_opt_valid.value_counts()
    .sort_index()
)

# ------------------------------------------------------------
# 5. TRAIN USING OPTUNA BEST PARAMETERS
# ------------------------------------------------------------

reproduction_model = XGBClassifier(
    **best_params,
    objective="binary:logistic",
    eval_metric="logloss",
    random_state=42,
    n_jobs=-1
)

reproduction_model.fit(
    X_opt_train,
    y_opt_train
)

# ------------------------------------------------------------
# 6. PREDICT VALIDATION SET
# ------------------------------------------------------------

opt_valid_probability = (
    reproduction_model
    .predict_proba(X_opt_valid)[:, 1]
)

opt_valid_prediction = (
    reproduction_model
    .predict(X_opt_valid)
)

# ------------------------------------------------------------
# 7. CALCULATE PERFORMANCE
# ------------------------------------------------------------

reproduced_auc = roc_auc_score(
    y_opt_valid,
    opt_valid_probability
)

reproduced_accuracy = accuracy_score(
    y_opt_valid,
    opt_valid_prediction
)

reproduced_precision = precision_score(
    y_opt_valid,
    opt_valid_prediction,
    zero_division=0
)

reproduced_recall = recall_score(
    y_opt_valid,
    opt_valid_prediction,
    zero_division=0
)

reproduced_f1 = f1_score(
    y_opt_valid,
    opt_valid_prediction,
    zero_division=0
)

# ------------------------------------------------------------
# 8. DISPLAY RESULTS
# ------------------------------------------------------------

print("\n" + "=" * 80)
print("REPRODUCED OPTUNA-STYLE VALIDATION PERFORMANCE")
print("=" * 80)

print(
    f"ROC-AUC  : {reproduced_auc:.4f}"
)

print(
    f"Accuracy : {reproduced_accuracy:.4f}"
)

print(
    f"Precision: {reproduced_precision:.4f}"
)

print(
    f"Recall   : {reproduced_recall:.4f}"
)

print(
    f"F1       : {reproduced_f1:.4f}"
)

# ------------------------------------------------------------
# 9. COMPARE AGAINST ORIGINAL OPTUNA RESULT
# ------------------------------------------------------------

print("\n" + "=" * 80)
print("COMPARISON WITH PREVIOUS OPTUNA RESULT")
print("=" * 80)

print(
    "Previous Optuna ROC-AUC : 0.9410"
)

print(
    f"Reproduced ROC-AUC      : "
    f"{reproduced_auc:.4f}"
)

print(
    f"Difference              : "
    f"{0.9410 - reproduced_auc:.4f}"
)

print(
    "\n✓ Optuna validation reproduction completed."
)

NOTEBOOK 9 — REPRODUCING OPTUNA VALIDATION SCORE

Loading:
..\data\processed\X_train_smotenc.csv
..\data\processed\y_train_smotenc.csv

Balanced training shape:
(658, 58)

Balanced class distribution:
RISK_LABEL
0    329
1    329
Name: count, dtype: int64

✓ All 47 SHAP-selected features are present.

SHAP-selected balanced training shape:
(658, 47)



Optuna-style training shape:
(526, 47)
Optuna-style validation shape:
(132, 47)

Training class distribution:
RISK_LABEL
0    263
1    263
Name: count, dtype: int64

Validation class distribution:
RISK_LABEL
0    66
1    66
Name: count, dtype: int64

REPRODUCED OPTUNA-STYLE VALIDATION PERFORMANCE
ROC-AUC  : 0.9410
Accuracy : 0.8788
Precision: 0.8676
Recall   : 0.8939
F1       : 0.8806

COMPARISON WITH PREVIOUS OPTUNA RESULT
Previous Optuna ROC-AUC : 0.9410
Reproduced ROC-AUC      : 0.9410
Difference              : -0.0000

✓ Optuna validation reproduction completed.


In [39]:
# ============================================================
# NOTEBOOK 9 — CELL 20
# SHAP FEATURE-SELECTION PROVENANCE CHECK
# ============================================================

print("=" * 80)
print("NOTEBOOK 9 — SHAP FEATURE-SELECTION PROVENANCE")
print("=" * 80)

print("\nNumber of SHAP-selected features:")
print(len(selected_features))

print("\nSelected features:")
for i, feature in enumerate(
    selected_features,
    start=1
):
    print(f"{i:02d}. {feature}")

print("\n" + "=" * 80)
print("DATA USED FOR SHAP SELECTION")
print("=" * 80)

print(
    "Original training observations:",
    X_train_clean.shape[0]
)

print(
    "Original predictors:",
    X_train_clean.shape[1]
)

print(
    "SMOTENC-balanced observations:",
    X_train_smotenc.shape[0]
    if "X_train_smotenc" in globals()
    else "Variable not currently loaded"
)

print(
    "SMOTENC-balanced predictors:",
    X_train_smotenc.shape[1]
    if "X_train_smotenc" in globals()
    else "Variable not currently loaded"
)

print("\n" + "=" * 80)
print("SHAP FEATURE COUNT")
print("=" * 80)

print(
    "Original features : 58"
)

print(
    "Selected features :",
    len(selected_features)
)

print(
    "Removed features  :",
    58 - len(selected_features)
)

print("\n✓ SHAP provenance diagnostic completed.")

NOTEBOOK 9 — SHAP FEATURE-SELECTION PROVENANCE

Number of SHAP-selected features:
47

Selected features:
01. GPA_S1
02. Hopeless/unmotivated
03. Sleep_hrs
04. LAB_AVG
05. Programme prepares for career
06. Concentration in self-study
07. Adequate supervision
08. CA_AVG
09. Financial_diff
10. Employment_hrs
11. Can improve performance
12. Self-motivates
13. Sense of belonging
14. Manages time effectively
15. Early intervention provided
16. Persists when difficult
17. Peer study groups effective
18. Anxious about assessments
19. Takes rest breaks
20. Seeks help when stuck
21. Regular exercise
22. Participates in class
23. Considered break
24. Age_group
25. Understands content pre-exam
26. CLIN_AVG
27. Takes organised notes
28. Physical health affected
29. ATT_RATE
30. Lecturers approachable
31. Sets academic goals
32. Study_hrs_day
33. Completes readings
34. EXAM_AVG
35. Confident in clinical skills
36. Prepared for clinical assess
37. Rotations affect performance
38. Self_risk_percep
39.